# ask

> answer from the vault, with citations back into it

In [ ]:
#| default_exp ask

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import *

[rishi](https://github.com/vedicreader/rishi) picks its backend from the *shape* of a model id —
`litert-community/…` or `.litertlm` goes to LiteRT, `.gguf` to llama.cpp, `mlx-community/…` to MLX,
a hosted name like `claude-sonnet-5` to fastllm — and raises rather than guessing when the id says
nothing. `MODELS` exists only so the common cases are one short word.

In [ ]:
#| export
import os, re
from fastcore.all import AttrDict, L, patch
from vishalakshi.core import Vault, tidy_bc

In [ ]:
#| export
VAULT_SP = """You answer questions from a personal research vault.

You are given numbered sections retrieved from the user's own corpus — papers, web pages,
transcripts, files and their own notes. Answer only from those sections.

Rules:
- Cite every claim with the bracketed number of the section it came from, like [2]. A sentence
  drawing on two sections cites both.
- If the sections do not answer the question, say exactly what is missing rather than filling the
  gap from memory. A vault that admits a hole is useful; one that guesses is not.
- Sections marked RELATED were reached by association, not by matching the question. Use them for
  context or to point somewhere worth reading next, and say so when you do.
- Prefer the user's own notes when they conflict with a source, and flag the disagreement."""

MODELS = {
    # alias          (model id,                                       runtime, note)
    'gemma-e2b':     ('litert-community/gemma-4-E2B-it-litert-lm',    'litert', 'default: ~2GB, CPU, runs anywhere'),
    'gemma-e4b':     ('litert-community/gemma-4-E4B-it-litert-lm',    'litert', 'larger LiteRT build'),
    'gemma-12b':     ('litert-community/gemma-4-12B-it-litert-lm',    'litert', 'biggest LiteRT build'),
    'qwen-4b-mlx':   ('mlx-community/Qwen3-4B-4bit',                  'mlx',    'Apple silicon'),
    'gemma-4b-mlx':  ('mlx-community/gemma-3-4b-it-4bit',             'mlx',    'Apple silicon'),
    'qwen-4b-gguf':  ('Qwen/Qwen3-4B-GGUF',                           'llama',  'llama.cpp, needs rishi[llama]'),
    'gemma-4b-gguf': ('ggml-org/gemma-3-4b-it-GGUF',                  'llama',  'llama.cpp, needs rishi[llama]'),
    'sonnet':        ('claude-sonnet-5',                              'remote', 'hosted, needs ANTHROPIC_API_KEY'),
    'opus':          ('claude-opus-5',                                'remote', 'hosted, needs ANTHROPIC_API_KEY'),
}

dflt_model = os.getenv('VISHALAKSHI_MODEL') or MODELS['gemma-e2b'][0]

def resolve_model(model:str=None, runtime:str=None) -> tuple:
    "`(model_id, runtime)` from a MODELS alias, a full id, or a `runtime/model` string."
    from rishi.core import resolve_runtime
    mid, rt = MODELS.get(model, (model or dflt_model, None))[:2]
    rt, mid = resolve_runtime(mid, runtime or rt)
    return mid, rt

In [ ]:
#| export
def mk_prompt(question:str,        # what you want to know
              ctx,                 # AttrDict from Vault.context()
              max_chars:int=4000,  # chars kept per section
              related:bool=True,   # include the associative leg
) -> str:
    """The user turn: the numbered sections, then the question.

    The numbering is the contract with the model: `[n]` in the answer maps to `ctx.results[n-1]`,
    which is what makes an answer checkable against the vault instead of merely plausible."""
    def sec(i, r):
        pg = f', pages {r.pages[0]}–{r.pages[1]}' if r.pages and r.pages[0] is not None else ''
        return f"[{i}] {tidy_bc(r.breadcrumb)}\n(source: {r.filename or r.doc_id}{pg})\n\n{(r.text or '')[:max_chars]}"
    parts = L(sec(i, r) for i, r in enumerate(ctx.results, 1))
    if related and ctx.related:
        parts.append('RELATED — not retrieved by the question, but connected to what was:\n' +
                     '\n'.join(f'- {tidy_bc(r.breadcrumb)} (reached by {r.via})' for r in ctx.related))
    if not parts: return f'The vault returned nothing for this question.\n\nQuestion: {question}'
    return '\n\n---\n\n'.join(parts) + f'\n\n---\n\nQuestion: {question}'

def cited(answer:str, results) -> L:
    'The sections an answer actually cited, in citation order — the audit trail for a claim.'
    ns = dict.fromkeys(int(m) for m in re.findall(r'\[(\d+)\]', answer or ''))
    def one(n, r): return dict(n=n, node_id=r.node_id, title=r.title, source=r.filename,
                               breadcrumb=tidy_bc(r.breadcrumb), doc_id=r.doc_id)
    return L(one(n, results[n-1]) for n in ns if 0 < n <= len(results))

In [ ]:
#| export
@patch
def chat(self:Vault, model:str=None, runtime:str=None, sp:str=VAULT_SP, **kw):
    """A `rishi.Chat` bound to this vault's system prompt, cached on the vault.

    Four backends, one call: `litert` (CPU, no API key, the default because it runs anywhere),
    `mlx` (Apple silicon), `llama` (any GGUF, needs `rishi[llama]`) and `remote` (hosted, via
    fastllm). Pass `runtime=` only when the model id cannot say for itself."""
    from rishi.core import Chat
    mid, rt = resolve_model(model, runtime)
    if getattr(self, '_chat_key', None) != (key := (mid, rt, sp)):
        self._chat, self._chat_key = Chat(mid, runtime=rt, sp=sp, **kw), key
    return self._chat

@patch
def ask(self:Vault,
        question:str,          # what you want to know
        model:str=None,        # a MODELS alias, a full id, or None -> $VISHALAKSHI_MODEL
        runtime:str=None,      # 'litert' | 'mlx' | 'llama' | 'remote'; inferred from the id if None
        sections:int=6,        # operative sections retrieved
        related:int=6,         # associative sections offered as leads
        kind:str=None,         # restrict retrieval to one or more KINDS
        max_chars:int=4000,    # chars of each section shown to the model
        sp:str=VAULT_SP,       # system prompt
        fresh:bool=True,       # start a new conversation rather than continuing the last
        **kw                   # forwarded to Vault.context
) -> AttrDict:
    """Retrieve, then answer with citations back into the vault.

    `cited` resolves the `[n]` markers in the answer back to `node_id`s you can `read()`, so every
    claim is one call away from the text it came from; `context` is the full retrieval, kept so you
    can inspect what the model was and was not shown."""
    from rishi.core import resp_text
    ctx = self.context(question, sections=sections, related=related, kind=kind, **kw)
    prompt = mk_prompt(question, ctx, max_chars=max_chars, related=bool(related))
    ch = self.chat(model=model, runtime=runtime, sp=sp)
    if fresh: ch.hist = []
    answer = resp_text(ch(prompt))
    mid, rt = resolve_model(model, runtime)
    return AttrDict(question=question, answer=answer, cited=cited(answer, ctx.results), context=ctx,
                    prompt=prompt, encoder=self.enc.note, model=mid, runtime=rt,
                    usage=getattr(ch, 'use', None))

@patch
def explain(self:Vault, node_id:str, model:str=None, max_chars:int=6000, **kw) -> AttrDict:
    'Have a model explain one section in the context of what the vault connects it to.'
    from rishi.core import resp_text
    sec, rel = self.read(node_id, max_chars=max_chars), self.related(node_id, limit=6)
    ch = self.chat(model=model, **kw); ch.hist = []
    prompt = (f"Section: {sec.get('title','')}\n\n{sec.get('text','')}\n\nOther sections in the vault "
              f"that read like it:\n" + '\n'.join(f"- {r['breadcrumb']}" for r in rel) +
              "\n\nExplain this section, then say what the related sections add or contradict.")
    return AttrDict(node_id=node_id, answer=resp_text(ch(prompt)), section=sec, related=rel)

## Try it

Retrieval needs no model; only the answering step does. `mk_prompt` is the whole contract, so it is
worth looking at what the model actually sees.

In [ ]:
v = Vault(':memory:', offline=True)
v.note('Late chunking beats naive chunking because context survives the split.')
print(mk_prompt('why late chunking?', v.context('late chunking'))[:400])

In [ ]:
test_eq(resolve_model('sonnet'), ('claude-sonnet-5', 'remote'))
test_eq(resolve_model('mlx-community/Qwen3-4B-4bit'), ('mlx-community/Qwen3-4B-4bit', 'mlx'))
test_eq(resolve_model('my-local.gguf', 'llama'), ('my-local.gguf', 'llama'))
test_fail(lambda: resolve_model('gemma-3-4b-it-int4'), contains='backend')

In [ ]:
res = L([AttrDict(node_id='d#1', title='A', breadcrumb='A › B', filename='f', doc_id='d')])
test_eq(cited('as [1] shows, and again [1], but not [9]', res).attrgot('node_id'), ['d#1'])